# P&P Full Experimentation — LIBERO-PRO

Two-phase evaluation (uncertainty measurement → refinement) across
position-perturbation and distractor suites, followed by a detailed analysis section.

**Run order — first session:** 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13 → 14

**Run order — subsequent sessions:** 1 → 2 → 3b → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13 → 14

---
## 1. GPU Check & Drive Mount
Run every session.

In [ ]:
import subprocess, torch
print(subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True).stdout.strip())
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE      = '/content/drive/MyDrive'
CACHE_DIR  = f'{DRIVE}/cs159_jeff/smolvla_colab_cache'
RESULTS_DIR = f'{DRIVE}/cs159_jeff/libero_pro_results'
DB_PATH    = f'{RESULTS_DIR}/rollouts_jeff_2_multimodal.db'
HF_HOME = f'{CACHE_DIR}/hf_models'
# SNAPSHOT   = f'{CACHE_DIR}/site_packages.tar.gz'
# LIBERO_TAR = f'{CACHE_DIR}/libero_pro_files.tar.gz'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

os.environ['HF_HOME']               = HF_HOME
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MUJOCO_GL']             = 'egl'

SNAPSHOT  = f'{CACHE_DIR}/site_packages.tar.gz'
PRO_CACHE = f'{CACHE_DIR}/libero_pro_files.tar.gz'

print(f'\nPackage snapshot: {"FOUND" if os.path.exists(SNAPSHOT) else "NOT FOUND — run Section 3"}')
print(f'LIBERO-PRO cache: {"FOUND" if os.path.exists(PRO_CACHE) else "NOT FOUND — run Section 5b"}')

NVIDIA L4, 23034 MiB
CUDA: True
GPU:  NVIDIA L4
VRAM: 23.7 GB
Mounted at /content/drive

Package snapshot: FOUND
LIBERO-PRO cache: FOUND


---
## 2. System Libraries
Run once per session (~2 min).

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq \
    libosmesa6-dev libgl1-mesa-glx libglfw3 libglew-dev \
    libegl1-mesa-dev patchelf ffmpeg
mkdir -p /usr/share/glvnd/egl_vendor.d
echo '{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}' \
    > /usr/share/glvnd/egl_vendor.d/10_nvidia.json
echo 'System deps done'

Selecting previously unselected package libglx-dev:amd64.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../00-libglx-dev_1.4.0-1_amd64.deb ...
Unpacking libglx-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libgl-dev:amd64.
Preparing to unpack .../01-libgl-dev_1.4.0-1_amd64.deb ...
Unpacking libgl-dev:amd64 (1.4.0-1) ...
Selecting previously unselected package libegl-dev:amd64.
Preparing to unpack .../02-libegl-dev_1.4.0-1_amd64.deb ...
Unpacking libegl-dev:amd64 (1.4.0-1) ...
Preparing to unpack .../03-libegl-mesa0_23.2.1-1ubuntu3.1~22.04.4_amd64.deb ...
Unpacking libegl-mesa0:amd64 (23.2.1-1ubuntu3.1~22.04.4) over (23.2.1-1ubuntu3.1~22.04.3) ...
Preparing to unpack .../04-libgbm1_23.2.1-1ubuntu3.1~22.04.4_amd64.deb ...
Unpacking libgbm1:amd64 (23.2.1-1ubuntu3.1~22.04.4) over (23.2.1-1ubuntu3.1~22.04.3) ...
Preparing to unpack .../05-libgl1-mesa-dri_23.2.1-1ubuntu3.1~22.04.4_amd64.deb ...
Unpacking libgl1-mesa-dri:amd64 

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


---
## 3. Python Package Install & Snapshot
### 3a. Full install — FIRST SESSION ONLY
Skip to **3b** on subsequent sessions.

In [ ]:
# ── Full pip install (first session only) ────────────────────────────────────
import subprocess, os
CACHE_DIR = '/content/drive/MyDrive/smolvla_colab_cache'
SNAPSHOT  = f'{CACHE_DIR}/site_packages.tar.gz'

packages = [
    'git+https://github.com/huggingface/lerobot.git#egg=lerobot[smolvla]',
    'mujoco', 'robosuite', 'libero',
]
for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.run(['pip', 'install', '-q', pkg], check=True)

print('\nSaving snapshot to Drive...')
subprocess.run(
    ['tar', '-czf', SNAPSHOT,
     '-C', '/usr/local/lib',
     f'python3.{__import__("sys").version_info.minor}/dist-packages'],
    check=True)
print(f'Snapshot saved: {SNAPSHOT}')

### 3b. Fast restore from Drive snapshot — SUBSEQUENT SESSIONS

In [ ]:
import subprocess, sys, os, time, shutil, importlib

LOCAL_SNAPSHOT = '/content/site_packages_restore.tar.gz'

if not os.path.exists(SNAPSHOT):
    raise FileNotFoundError(
        'No snapshot found. Run Section 3a (full install) first to create it.'
    )

t0 = time.time()
size_mb = os.path.getsize(SNAPSHOT) / 1e6
print(f'Found snapshot on Drive: {size_mb:.0f} MB')

print('Copying snapshot from Drive to local disk (avoids Drive timeout bugs)...')
shutil.copy(SNAPSHOT, LOCAL_SNAPSHOT)

print('Extracting packages...')
result = subprocess.run(
    ['tar', '-xzf', LOCAL_SNAPSHOT, '-C', '/'],
    capture_output=True, text=True
)
os.remove(LOCAL_SNAPSHOT)

if result.returncode != 0:
    print('Restore failed:', result.stderr[:500])
else:
    elapsed = time.time() - t0
    print(f'Restored in {elapsed:.1f}s')

importlib.invalidate_caches()

for pkg in ['mujoco', 'robosuite', 'libero', 'lerobot']:
    try:
        importlib.import_module(pkg)
        print(f'  {pkg} OK')
    except ImportError as e:
        print(f'  {pkg} MISSING: {e}')
print('Restore complete.')

Found snapshot on Drive: 11198 MB
Copying snapshot from Drive to local disk (avoids Drive timeout bugs)...
Extracting packages...
Restored in 427.4s


[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


  mujoco OK
  robosuite OK
  libero OK
  lerobot OK
Restore complete.


---
## 4. Clone LIBERO-PRO Repo
Run every session (fast, just clones the repo without installing).

In [ ]:
import subprocess, os

REPO_DIR = '/content/LIBERO-PRO'
if not os.path.exists(REPO_DIR):
    print('Cloning LIBERO-PRO...')
    result = subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/Zxy-MLlab/LIBERO-PRO.git',
         REPO_DIR],
        capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Clone failed:\n{result.stderr}')
    print('Clone complete.')
else:
    print(f'Repo already exists at {REPO_DIR}')

# Verify bddl directory structure
bddl_root = f'{REPO_DIR}/libero/libero/bddl_files'
if os.path.exists(bddl_root):
    suites = sorted(os.listdir(bddl_root))
    print(f'LIBERO-PRO bddl suites found: {len(suites)}')
    pos_suites  = [s for s in suites if 'temp_' in s]
    with_suites = [s for s in suites if '_with_' in s]
    print(f'  Position perturbation: {pos_suites}')
    print(f'  Distractor (_with_*): {with_suites}')
else:
    print('WARNING: bddl_files not found in repo clone')

Cloning LIBERO-PRO...
Clone complete.
LIBERO-PRO bddl suites found: 52
  Position perturbation: ['libero_object_temp_x0.1', 'libero_object_temp_x0.2', 'libero_object_temp_x0.3', 'libero_object_temp_x0.4', 'libero_object_temp_x0.5', 'libero_object_temp_y0.1', 'libero_object_temp_y0.2', 'libero_object_temp_y0.3', 'libero_object_temp_y0.4', 'libero_object_temp_y0.5']
  Distractor (_with_*): ['libero_10_with_blue_stick', 'libero_10_with_diffpos_stick', 'libero_10_with_milk', 'libero_10_with_mug', 'libero_10_with_red_box', 'libero_10_with_red_stick', 'libero_goal_with_blue_stick', 'libero_goal_with_diffpos_stick', 'libero_goal_with_green_mug', 'libero_goal_with_milk', 'libero_goal_with_mug', 'libero_goal_with_red_box', 'libero_goal_with_red_stick', 'libero_goal_with_rotated_stick', 'libero_goal_with_yellow_book', 'libero_object_with_blue_stick', 'libero_object_with_diffpos_stick', 'libero_object_with_mug', 'libero_object_with_red_box', 'libero_object_with_red_stick', 'libero_object_with_tri

---
## 5. LIBERO-PRO Assets (bddl + init files)
### 5a. Download from HuggingFace & save to Drive cache — FIRST SESSION ONLY

In [ ]:
# # ── Download LIBERO-PRO bddl + init files from HuggingFace and cache ─────────
# # Repo: zhouxueyang/LIBERO-Pro (dataset) — contains bddl_files/ and init_files/
# import sys, os, tarfile, time
# from huggingface_hub import snapshot_download

# LIBERO_SITE = f'/usr/local/lib/python3.{sys.version_info.minor}/dist-packages/libero/libero'
# CACHE_DIR   = '/content/drive/MyDrive/smolvla_colab_cache'
# PRO_CACHE   = f'{CACHE_DIR}/libero_pro_files.tar.gz'
# HF_TMP      = '/content/libero_pro_hf'

# if os.path.exists(PRO_CACHE):
#     print(f'Cache already exists ({os.path.getsize(PRO_CACHE)//1024//1024} MB). '
#           f'Use Section 5b to restore.')
# else:
#     print('Downloading from HuggingFace (zhouxueyang/LIBERO-Pro)...')
#     t0 = time.time()
#     snapshot_download(
#         repo_id='zhouxueyang/LIBERO-Pro',
#         repo_type='dataset',
#         local_dir=HF_TMP,
#     )
#     print(f'Downloaded in {time.time()-t0:.0f}s. Compressing to Drive...')

#     t1 = time.time()
#     with tarfile.open(PRO_CACHE, 'w:gz') as tar:
#         for subdir in ['bddl_files', 'init_files']:
#             src = os.path.join(HF_TMP, subdir)
#             if os.path.exists(src):
#                 tar.add(src, arcname=subdir)
#                 print(f'  Packed {subdir}/')
#             else:
#                 print(f'  WARNING: {subdir} not found in download')
#     print(f'Saved to Drive in {time.time()-t1:.0f}s '
#           f'({os.path.getsize(PRO_CACHE)//1024//1024} MB)')

Cache already exists (0 MB). Use Section 5b to restore.


### 5b. Restore from Drive cache — SUBSEQUENT SESSIONS

In [ ]:
# ── Restore bddl + init files from Drive cache ────────────────────────────────
import sys, os, tarfile

LIBERO_SITE = f'/usr/local/lib/python3.{sys.version_info.minor}/dist-packages/libero/libero'
CACHE_DIR   = f'{DRIVE}/cs159_jeff/smolvla_colab_cache'
PRO_CACHE   = f'{CACHE_DIR}/libero_pro_files.tar.gz'

assert os.path.exists(PRO_CACHE), f'Cache not found: {PRO_CACHE} — run Section 5a first'
print('Restoring LIBERO-PRO files from Drive cache...')
with tarfile.open(PRO_CACHE, 'r:gz') as tar:
    tar.extractall(LIBERO_SITE)
print('Restore complete.')
print(f'  bddl_files: {len(os.listdir(os.path.join(LIBERO_SITE, "bddl_files")))} entries')
print(f'  init_files: {len(os.listdir(os.path.join(LIBERO_SITE, "init_files")))} entries')

Restoring LIBERO-PRO files from Drive cache...
Restore complete.
  bddl_files: 21 entries
  init_files: 21 entries


/tmp/ipykernel_5542/1872230037.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(LIBERO_SITE)


In [ ]:
# ── Pre-import libero benchmark using pip version before patching ─────────────
# Must happen before Section 6 copies LIBERO-PRO files over the pip versions.
# Python will cache this import; the patched files take effect only after
# the explicit reload in Section 10.
import libero.libero.benchmark as _bm_preload
print('libero.libero.benchmark pre-loaded (pip version cached).')

Do you want to specify a custom path for the dataset folder? (Y/N): N
Initializing the default config file...
The following information is stored in the config file: /root/.libero/config.yaml
benchmark_root: /usr/local/lib/python3.12/dist-packages/libero/libero
bddl_files: /usr/local/lib/python3.12/dist-packages/libero/libero/./bddl_files
init_states: /usr/local/lib/python3.12/dist-packages/libero/libero/./init_files
datasets: /usr/local/lib/python3.12/dist-packages/libero/libero/../datasets
assets: /usr/local/lib/python3.12/dist-packages/libero/libero/./assets
libero.libero.benchmark pre-loaded (pip version cached).


---
## 6. Merge LIBERO-PRO Files & Apply All Patches
Run every session. Copies bddl/init files from the repo, registers new suites, and applies
all compatibility patches (dynamo, torch.load, CUSTOM_KEY, object aliases, benchmark dict).

In [ ]:
import shutil, os, sys, re, importlib

LIBERO_SITE    = f'/usr/local/lib/python3.{sys.version_info.minor}/dist-packages/libero/libero'
LIBERO_PRO_DIR = '/content/LIBERO-PRO'

# ── Step 1: Copy patched files from LIBERO-PRO repo ──────────────────────────
files_to_patch = [
    'benchmark/__init__.py',
    'benchmark/libero_suite_task_map.py',
    'envs/objects/__init__.py',
]
for rel_path in files_to_patch:
    src = os.path.join(LIBERO_PRO_DIR, 'libero/libero', rel_path)
    dst = os.path.join(LIBERO_SITE, rel_path)
    if not os.path.exists(src):
        print(f'WARNING: not found in clone: {rel_path}')
        continue
    os.makedirs(os.path.dirname(dst), exist_ok=True)  # create benchmark/ if missing
    bak = dst + '.original_bak'
    if os.path.exists(dst) and not os.path.exists(bak):
        shutil.copy2(dst, bak)
    shutil.copy2(src, dst)
    print(f'Patched: {rel_path}')

# Copy any new object definition .py files LIBERO-PRO added
src_objects = os.path.join(LIBERO_PRO_DIR, 'libero/libero/envs/objects')
dst_objects = os.path.join(LIBERO_SITE, 'envs/objects')
for fname in os.listdir(src_objects):
    if not fname.endswith('.py'):
        continue
    dst_file = os.path.join(dst_objects, fname)
    src_file = os.path.join(src_objects, fname)
    if not os.path.exists(dst_file):
        shutil.copy2(src_file, dst_file)
        print(f'Added new object file: {fname}')

# ── Step 2: Fix libero_mine KeyError (.get() patch) ──────────────────────────
benchmark_init = os.path.join(LIBERO_SITE, 'benchmark/__init__.py')
with open(benchmark_init, 'r') as f:
    content = f.read()
old = 'for task in libero_task_map[libero_suite]:'
new = 'for task in libero_task_map.get(libero_suite, []):'
if old in content:
    content = content.replace(old, new)
    with open(benchmark_init, 'w') as f:
        f.write(content)
    print('Fixed: .get() patch applied')
elif new in content:
    print('Already fixed: .get() patch')
else:
    print('WARNING: expected line not found in benchmark/__init__.py')

# ── Step 3: Make libero_suites dynamic ───────────────────────────────────────
with open(benchmark_init, 'r') as f:
    content = f.read()
if 'libero_suites = list(libero_task_map.keys())' in content:
    print('Already patched: libero_suites dynamic')
else:
    match = re.search(r'libero_suites\s*=\s*[\[\(].*?[\]\)]', content, re.DOTALL)
    if match:
        content = content[:match.start()] + \
                  'libero_suites = list(libero_task_map.keys())' + \
                  content[match.end():]
        with open(benchmark_init, 'w') as f:
            f.write(content)
        print('Patched: libero_suites now dynamic')
    else:
        loop_match = re.search(r'for libero_suite in libero_suites:', content)
        if loop_match:
            insert_pos = loop_match.start()
            content = content[:insert_pos] + \
                      'libero_suites = list(libero_task_map.keys())\n' + \
                      content[insert_pos:]
            with open(benchmark_init, 'w') as f:
                f.write(content)
            print('Patched: libero_suites dynamic (fallback method)')
        else:
            print('ERROR: could not find patch point — check benchmark/__init__.py manually')

# ── Step 4: Add PRO suite entries to task map from bddl dirs ─────────────────
task_map_file = os.path.join(LIBERO_SITE, 'benchmark/libero_suite_task_map.py')
PRO_SUFFIXES  = ('_lan', '_swap', '_object', '_task', '_env', '_temp')
BDDL_DIR      = os.path.join(LIBERO_SITE, 'bddl_files')

with open(task_map_file, 'r') as f:
    existing = f.read()

additions = []
for suite_dir in sorted(os.listdir(BDDL_DIR)):
    if not any(suite_dir.endswith(sfx) for sfx in PRO_SUFFIXES):
        continue
    bddl_suite_dir = os.path.join(BDDL_DIR, suite_dir)
    tasks = sorted([f.replace('.bddl', '') for f in os.listdir(bddl_suite_dir)
                    if f.endswith('.bddl')])
    if not tasks:
        continue
    has_tasks = bool(re.search(
        rf'"{re.escape(suite_dir)}"\s*[:\]]\s*\[.*?\S.*?\]',
        existing, re.DOTALL))
    if has_tasks:
        continue
    additions.append((suite_dir, tasks))

if additions:
    with open(task_map_file, 'a') as f:
        f.write('\n\n# ── LIBERO-PRO standard suites (built from bddl files) ──\n')
        for suite_name, tasks in additions:
            f.write(f'libero_task_map["{suite_name}"] = [\n')
            for task in tasks:
                f.write(f'    "{task}",\n')
            f.write(']\n\n')
    print(f'Added {len(additions)} PRO suite(s) to task map')
else:
    print('Task map: all PRO suites already registered')

# ── Step 5: Register position perturbation suites in task map ────────────────
TEMP_TASKS = [
    'pick_up_the_alphabet_soup_and_place_it_in_the_basket',
    'pick_up_the_bbq_sauce_and_place_it_in_the_basket',
    'pick_up_the_butter_and_place_it_in_the_basket',
    'pick_up_the_chocolate_pudding_and_place_it_in_the_basket',
    'pick_up_the_cream_cheese_and_place_it_in_the_basket',
    'pick_up_the_ketchup_and_place_it_in_the_basket',
    'pick_up_the_milk_and_place_it_in_the_basket',
    'pick_up_the_orange_juice_and_place_it_in_the_basket',
    'pick_up_the_salad_dressing_and_place_it_in_the_basket',
    'pick_up_the_tomato_sauce_and_place_it_in_the_basket',
]
TEMP_STRENGTHS = (
    [f'libero_object_temp_x{v}' for v in ['0.1','0.2','0.3','0.4','0.5']] +
    [f'libero_object_temp_y{v}' for v in ['0.1','0.2','0.3','0.4','0.5']]
)
with open(task_map_file, 'r') as f:
    existing = f.read()
temp_additions = [s for s in TEMP_STRENGTHS if f'"{s}"' not in existing]
if temp_additions:
    with open(task_map_file, 'a') as f:
        f.write('\n\n# ── Position perturbation suites ──\n')
        for suite in temp_additions:
            f.write(f'libero_task_map["{suite}"] = [\n')
            for t in TEMP_TASKS:
                f.write(f'    "{t}",\n')
            f.write(']\n')
    print(f'Added {len(temp_additions)} position perturbation suites')
else:
    print('Position perturbation suites already registered')

# ── Step 6: Copy custom assets for _with_milk suites ─────────────────────────
REPO_ASSETS = f'{LIBERO_PRO_DIR}/notebooks/custom_assets'
DIST_ASSETS = f'/usr/local/lib/python3.{sys.version_info.minor}/dist-packages/notebooks/custom_assets'
if os.path.exists(REPO_ASSETS) and not os.path.exists(DIST_ASSETS):
    shutil.copytree(REPO_ASSETS, DIST_ASSETS)
    print('Copied custom_assets to dist-packages')


print('\nSection 6 complete. Run torch patches + policy load next, then Section 10 will reload benchmark.')

Patched: benchmark/__init__.py
Patched: benchmark/libero_suite_task_map.py
Patched: envs/objects/__init__.py
Added new object file: self_designed_object.py
Fixed: .get() patch applied
Patched: libero_suites now dynamic
Task map: all PRO suites already registered
Added 10 position perturbation suites
Copied custom_assets to dist-packages

Section 6 complete. Run torch patches + policy load next, then Section 10 will reload benchmark.


---
## 7. Load Policy (pi0.5)
Run every session.

In [ ]:
import os, torch, numpy as np, time, json, math
from transformers import AutoTokenizer

# ── 1. Globally disable torch.dynamo ─────────────────────────────────────────
os.environ['TORCHDYNAMO_DISABLE'] = '1'
import torch._dynamo
torch._dynamo.config.disable = True
print('torch.dynamo disabled globally.')

# ── 2. Restore any previous torch.load patch ─────────────────────────────────
if hasattr(torch, '_libero_true_load'):
    torch.load = torch._libero_true_load

# ── 3. Patch torch.ao.quantization constants ─────────────────────────────────
import torch.ao.quantization as _taoq
for name, val in [('CUSTOM_KEY', 'custom'),
                  ('NUMERIC_DEBUG_HANDLE_KEY', 'numeric_debug_handle')]:
    if not hasattr(_taoq, name):
        setattr(_taoq, name, val)
        print(f'Patched torch.ao.quantization.{name}')

# ── 4. Fix torch.load for LIBERO .pruned_init files ──────────────────────────
try:
    torch.serialization.add_safe_globals([
        np.core.multiarray._reconstruct,
        np.ndarray, np.dtype,
        np.core.multiarray.scalar,

    ])
except Exception:
    pass

if not hasattr(torch, '_libero_true_load'):
    torch._libero_true_load = torch.load
_true = torch._libero_true_load

def _patched_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _true(*args, **kwargs)

torch.load = _patched_load
print('torch.load patched (weights_only=False default).')

# ── 5. Load policy ────────────────────────────────────────────────────────────
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv
from lerobot.policies.factory import make_pre_post_processors
from huggingface_hub import login

login()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

policy = PI05Policy.from_pretrained('lerobot/pi05_libero_finetuned').to(device).eval()
tokenizer = AutoTokenizer.from_pretrained('google/paligemma-3b-pt-224')
print(f'PI05 loaded — {sum(p.numel() for p in policy.parameters())/1e6:.0f}M params on {device}')

CAMERAS             = ['agentview', 'robot0_eye_in_hand']
IMG_SIZE            = 360
LIBERO_DUMMY_ACTION = [0.0] * 6 + [-1.0]
NUM_STEPS_WAIT      = 10

preprocess, postprocess = make_pre_post_processors(
    policy.config,
    'lerobot/pi05_libero_finetuned',
    preprocessor_overrides={'device_processor': {'device': str(device)}},
)

def _quat2axisangle(quat):
    if quat[3] > 1.0:  quat[3] = 1.0
    elif quat[3] < -1.0: quat[3] = -1.0
    den = np.sqrt(1.0 - quat[3] ** 2)
    if math.isclose(den, 0.0): return np.zeros(3)
    return (quat[:3] * 2.0 * math.acos(quat[3])) / den

def obs_to_policy(obs_dict, task_desc, device):
    agentview = np.ascontiguousarray(obs_dict['agentview_image'][::-1, ::-1])
    wrist      = np.ascontiguousarray(obs_dict['robot0_eye_in_hand_image'][::-1, ::-1])
    img_agent  = torch.from_numpy(agentview / 255.0).permute(2,0,1).float()
    img_wrist  = torch.from_numpy(wrist     / 255.0).permute(2,0,1).float()
    state = np.concatenate([
        obs_dict['robot0_eef_pos'],
        _quat2axisangle(obs_dict['robot0_eef_quat']),
        obs_dict['robot0_gripper_qpos'],
    ])
    return {
        'observation.images.image':  img_agent,
        'observation.images.image2': img_wrist,
        'observation.state': torch.from_numpy(state).float(),
        'task': task_desc,
    }

def run_episode(env, init_state, policy, task_desc, max_steps, device):
    env.reset(); policy.reset()
    obs = env.set_init_state(init_state)
    for _ in range(NUM_STEPS_WAIT):
        obs, _, _, _ = env.step(LIBERO_DUMMY_ACTION)
    t0 = time.time()
    for step in range(max_steps):
        raw_obs = obs_to_policy(obs, task_desc, device)
        batch   = preprocess(raw_obs)
        with torch.no_grad():
            action = policy.select_action(batch)
        action = postprocess(action)
        if isinstance(action, torch.Tensor):
            action = action.squeeze(0).cpu().numpy()
        obs, _, done, _ = env.step(action)
        if env.check_success():
            return True, step+1, time.time()-t0
        if done:
            break
    return False, step+1, time.time()-t0

print('Eval helpers defined.')

torch.dynamo disabled globally.
Patched torch.ao.quantization.CUSTOM_KEY
Patched torch.ao.quantization.NUMERIC_DEBUG_HANDLE_KEY
torch.load patched (weights_only=False default).


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


The PI05 model is a direct port of the OpenPI implementation. 
This implementation follows the original OpenPI structure for compatibility. 
Original implementation: https://github.com/Physical-Intelligence/openpi
Loading model from: lerobot/pi05_libero_finetuned


✓ Loaded state dict from model.safetensors
All keys loaded successfully!
PI05 loaded — 4143M params on cuda
Eval helpers defined.


---
## 8. Generate Init States for Position Perturbation Suites
Must run once per session (or once ever if you cache results to Drive).
Regenerates `.pruned_init` files using the perturbed bddl positions.

In [ ]:
import shutil, os, sys

import numpy as np, random
np.random.seed(42)
random.seed(42)

LIBERO_SITE  = f'/usr/local/lib/python3.{sys.version_info.minor}/dist-packages/libero/libero'
BDDL_DST     = os.path.join(LIBERO_SITE, 'bddl_files')
INIT_DST     = os.path.join(LIBERO_SITE, 'init_files')
REPO_BDDL    = '/content/LIBERO-PRO/libero/libero/bddl_files'
LIBERO_OBJ_INIT = os.path.join(INIT_DST, 'libero_object')

STRENGTHS = [f'libero_object_temp_x{v}' for v in ['0.1','0.2','0.3','0.4','0.5']] + \
            [f'libero_object_temp_y{v}' for v in ['0.1','0.2','0.3','0.4','0.5']]

for suite in STRENGTHS:
    src_bddl = os.path.join(REPO_BDDL, suite)
    dst_bddl = os.path.join(BDDL_DST, suite)
    dst_init = os.path.join(INIT_DST, suite)

    if not os.path.exists(src_bddl):
        print(f'  SKIP (not in repo): {suite}')
        continue

    # Copy bddl files
    if not os.path.exists(dst_bddl):
        shutil.copytree(src_bddl, dst_bddl)
        print(f'  Copied bddl: {suite}')
    else:
        print(f'  bddl already exists: {suite}')

    # Copy init files from libero_object (same tasks, same robot start state)
    if not os.path.exists(dst_init):
        shutil.copytree(LIBERO_OBJ_INIT, dst_init)
        print(f'  Copied init files from libero_object → {suite}')
    else:
        print(f'  init already exists: {suite}')

print('\nDone.')

  Copied bddl: libero_object_temp_x0.1
  Copied init files from libero_object → libero_object_temp_x0.1
  Copied bddl: libero_object_temp_x0.2
  Copied init files from libero_object → libero_object_temp_x0.2
  Copied bddl: libero_object_temp_x0.3
  Copied init files from libero_object → libero_object_temp_x0.3
  Copied bddl: libero_object_temp_x0.4
  Copied init files from libero_object → libero_object_temp_x0.4
  Copied bddl: libero_object_temp_x0.5
  Copied init files from libero_object → libero_object_temp_x0.5
  Copied bddl: libero_object_temp_y0.1
  Copied init files from libero_object → libero_object_temp_y0.1
  Copied bddl: libero_object_temp_y0.2
  Copied init files from libero_object → libero_object_temp_y0.2
  Copied bddl: libero_object_temp_y0.3
  Copied init files from libero_object → libero_object_temp_y0.3
  Copied bddl: libero_object_temp_y0.4
  Copied init files from libero_object → libero_object_temp_y0.4
  Copied bddl: libero_object_temp_y0.5
  Copied init files from 

Uncomment to generate init states from scratch (load from Drive instead of files already exist)

In [ ]:
# import numpy as np, torch, os, sys
# from libero.libero.envs import OffScreenRenderEnv

# LIBERO_SITE = f'/usr/local/lib/python3.{sys.version_info.minor}/dist-packages/libero/libero'
# CAMERAS     = ['agentview', 'robot0_eye_in_hand']
# IMG_SIZE    = 360
# N_STATES    = 10

# TARGET_TEMP_SUITES = [
#     'libero_object_temp_x0.1',
#     'libero_object_temp_y0.1',
#     'libero_object_temp_y0.2',
#     'libero_object_temp_x0.2',
# ]

# TASKS = [
#     'pick_up_the_alphabet_soup_and_place_it_in_the_basket',
#     'pick_up_the_bbq_sauce_and_place_it_in_the_basket',
#     'pick_up_the_butter_and_place_it_in_the_basket',
#     'pick_up_the_chocolate_pudding_and_place_it_in_the_basket',
#     'pick_up_the_cream_cheese_and_place_it_in_the_basket',
#     'pick_up_the_ketchup_and_place_it_in_the_basket',
#     'pick_up_the_milk_and_place_it_in_the_basket',
#     'pick_up_the_orange_juice_and_place_it_in_the_basket',
#     'pick_up_the_salad_dressing_and_place_it_in_the_basket',
#     'pick_up_the_tomato_sauce_and_place_it_in_the_basket',
# ]

# for suite in TARGET_TEMP_SUITES:
#     save_dir = os.path.join(LIBERO_SITE, 'init_files', suite)
#     os.makedirs(save_dir, exist_ok=True)
#     print(f'\nGenerating init states for {suite}...')

#     for task in TASKS:
#         bddl_path = os.path.join(LIBERO_SITE, 'bddl_files', suite, task + '.bddl')
#         save_path = os.path.join(save_dir, task + '.pruned_init')

#         # Force overwrite — delete any existing placeholder
#         if os.path.exists(save_path):
#             os.remove(save_path)

#         env = OffScreenRenderEnv(
#             bddl_file_name=bddl_path,
#             camera_names=CAMERAS,
#             camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
#             has_offscreen_renderer=True, use_camera_obs=True,
#             has_renderer=False, reward_shaping=False,
#         )
#         states = []
#         for _ in range(N_STATES):
#             env.reset()
#             states.append(env.sim.get_state().flatten())
#         env.close()

#         states_arr = np.array(states)
#         torch.save(states_arr, save_path)
#         print(f'  {task}: shape={states_arr.shape}')

# print('\nDone.')


Generating init states for libero_object_temp_x0.1...
Local assets not found. Downloading from HuggingFace Hub...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 586 files:   0%|          | 0/586 [00:00<?, ?it/s]

Assets downloaded successfully to /root/.cache/libero/assets
  pick_up_the_alphabet_soup_and_place_it_in_the_basket: shape=(10, 110)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  pick_up_the_bbq_sauce_and_place_it_in_the_basket: shape=(10, 110)
  pick_up_the_butter_and_place_it_in_the_basket: shape=(10, 110)
  pick_up_the_chocolate_pudding_and_place_it_in_the_basket: shape=(10, 110)
  pick_up_the_cream_cheese_and_place_it_in_the_basket: shape=(10, 110)
  pick_up_the_ketchup_and_place_it_in_the_basket: shape=(10, 110)
  pick_up_the_milk_and_place_it_in_the_basket: shape=(10, 110)
  pick_up_the_orange_juice_and_place_it_in_the_basket: shape=(10, 110)
  pick_up_the_salad_dressing_and_place_it_in_the_basket: shape=(10, 110)
  pick_up_the_tomato_sauce_and_place_it_in_the_basket: shape=(10, 110)

Generating init states for libero_object_temp_y0.1...
  pick_up_the_alphabet_soup_and_place_it_in_the_basket: shape=(10, 110)
  pick_up_the_bbq_sauce_and_place_it_in_the_basket: shape=(10, 110)
  pick_up_the_butter_and_place_it_in_the_basket: shape=(10, 110)
  pick_up_the_chocolate_pudding_and_place_it_in_the_basket: shape=(10, 110)
  pick_up_the_cream_cheese_and_place_it_

---
## 9. Generate Init States for Distractor Suites
Run once (idempotent — skips tasks already done).

In [ ]:
# import shutil, os, sys, importlib
# import numpy as np, torch
# from libero.libero.envs import OffScreenRenderEnv

# LIBERO_SITE   = f'/usr/local/lib/python3.{sys.version_info.minor}/dist-packages/libero/libero'
# REPO_BDDL     = '/content/LIBERO-PRO/libero/libero/bddl_files'
# task_map_file = os.path.join(LIBERO_SITE, 'benchmark/libero_suite_task_map.py')
# BDDL_DIR      = os.path.join(LIBERO_SITE, 'bddl_files')
# CAMERAS       = ['agentview', 'robot0_eye_in_hand']
# IMG_SIZE      = 360
# N_STATES      = 10

# WITH_SUITES = [
#     'libero_spatial_with_milk',
#     # 'libero_spatial_with_yellow_book',
#     # 'libero_goal_with_milk',
#     'libero_goal_with_yellow_book',
#     # 'libero_object_with_yellow_book',
#     # 'libero_10_with_milk',
# ]

# # ── Step 1: Copy bddl files from repo ────────────────────────────────────────
# for suite in WITH_SUITES:
#     src = os.path.join(REPO_BDDL, suite)
#     dst = os.path.join(BDDL_DIR, suite)
#     if not os.path.exists(src):
#         print(f'  SKIP (not in repo): {suite}')
#         continue
#     if not os.path.exists(dst):
#         shutil.copytree(src, dst)
#         print(f'  Copied bddl: {suite}')
#     else:
#         print(f'  bddl already exists: {suite}')

# # ── Step 2: Generate init states ─────────────────────────────────────────────
# for suite in WITH_SUITES:
#     bddl_dir = os.path.join(BDDL_DIR, suite)
#     save_dir = os.path.join(LIBERO_SITE, 'init_files', suite)
#     if not os.path.exists(bddl_dir):
#         print(f'  SKIP (no bddl dir): {suite}')
#         continue
#     os.makedirs(save_dir, exist_ok=True)
#     tasks = sorted([f.replace('.bddl','') for f in os.listdir(bddl_dir) if f.endswith('.bddl')])
#     print(f'\nGenerating init states for {suite} ({len(tasks)} tasks)...')
#     for task in tasks:
#         bddl_path = os.path.join(bddl_dir, task + '.bddl')
#         save_path = os.path.join(save_dir, task + '.pruned_init')
#         if os.path.exists(save_path):
#             print(f'  SKIP (exists): {task[:55]}')
#             continue
#         try:
#             env = OffScreenRenderEnv(
#                 bddl_file_name=bddl_path,
#                 camera_names=CAMERAS,
#                 camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
#                 has_offscreen_renderer=True, use_camera_obs=True,
#                 has_renderer=False, reward_shaping=False,
#             )
#             states = []
#             for _ in range(N_STATES):
#                 env.reset()
#                 states.append(env.sim.get_state().flatten())
#             env.close()
#             states_arr = np.array(states)
#             torch.save(states_arr, save_path)
#             print(f'  {task[:55]}: {states_arr.shape}')
#         except Exception as e:
#             print(f'  ERROR {task[:45]}: {e}')

# # ── Step 3: Register in task map file ────────────────────────────────────────
# with open(task_map_file, 'r') as f:
#     existing = f.read()

# added = []
# with open(task_map_file, 'a') as f:
#     f.write('\n\n# ── _with_* suites ──\n')
#     for suite in WITH_SUITES:
#         if f'"{suite}"' in existing:
#             print(f'  {suite}: already in task map')
#             continue
#         bddl_suite_dir = os.path.join(BDDL_DIR, suite)
#         if not os.path.exists(bddl_suite_dir):
#             continue
#         tasks = sorted([t.replace('.bddl','') for t in os.listdir(bddl_suite_dir)
#                         if t.endswith('.bddl')])
#         f.write(f'libero_task_map["{suite}"] = [\n')
#         for t in tasks:
#             f.write(f'    "{t}",\n')
#         f.write(']\n\n')
#         added.append(suite)
#         print(f'  Added to task map: {suite} ({len(tasks)} tasks)')

# # ── Step 4: Reload task map and inject into benchmark.task_maps ──────────────
# import libero.libero.benchmark.libero_suite_task_map as _tm
# importlib.reload(_tm)
# import libero.libero.benchmark as _bm_with
# importlib.reload(_bm_with)

# from libero.libero.benchmark.libero_suite_task_map import libero_task_map
# from libero.libero.benchmark import grab_language_from_filename, Task

# for suite in WITH_SUITES:
#     if suite not in _bm_with.task_maps and suite in libero_task_map:
#         _bm_with.task_maps[suite] = {}
#         for task_name in libero_task_map[suite]:
#             try:
#                 language = grab_language_from_filename(task_name + '.bddl')
#                 _bm_with.task_maps[suite][task_name] = Task(
#                     name=task_name, language=language,
#                     problem_folder=suite, bddl_file=task_name + '.bddl')
#             except Exception as e:
#                 print(f'    WARNING task {task_name}: {e}')
#         print(f'  Injected {len(_bm_with.task_maps[suite])} tasks into task_maps[{suite}]')

# # ── Step 5: Extend monkey-patch to cover both temp and _with_* suites ─────────
# TEMP_STRENGTHS = (
#     [f'libero_object_temp_x{v}' for v in ['0.1','0.2','0.3','0.4','0.5']] +
#     [f'libero_object_temp_y{v}' for v in ['0.1','0.2','0.3','0.4','0.5']]
# )
# ALL_CUSTOM_SUITES = TEMP_STRENGTHS + WITH_SUITES

# _orig = _bm_with.get_benchmark_dict

# def _patched_get_benchmark_dict():
#     result = dict(_orig())
#     for suite in ALL_CUSTOM_SUITES:
#         if suite not in result and \
#            suite in _bm_with.task_maps and \
#            _bm_with.task_maps[suite]:
#             def make_cls(s):
#                 class _Suite:
#                     def __init__(self):
#                         self._tasks = list(_bm_with.task_maps[s].values())
#                         self.n_tasks = len(self._tasks)
#                     def get_task(self, i): return self._tasks[i]
#                     def get_task_names(self): return list(_bm_with.task_maps[s].keys())
#                     def get_task_init_states(self, i):
#                         import torch, os
#                         task = self._tasks[i]
#                         init_file = os.path.join(
#                             f'/usr/local/lib/python3.{sys.version_info.minor}'
#                             f'/dist-packages/libero/libero/init_files',
#                             task.problem_folder,
#                             task.bddl_file.replace('.bddl', '.pruned_init'))
#                         return torch.load(init_file)
#                 return _Suite
#             result[suite] = make_cls(suite)
#     return result

# _bm_with.get_benchmark_dict = _patched_get_benchmark_dict

# # ── Verify ────────────────────────────────────────────────────────────────────
# bm = _bm_with.get_benchmark_dict()
# for suite in WITH_SUITES:
#     if suite in bm:
#         n = bm[suite]().n_tasks
#         MAX_STEPS_MAP[suite] = 300
#         print(f'  {suite}: {n} tasks ✓')
#     else:
#         print(f'  {suite}: NOT in benchmark_dict ✗')

In [ ]:
# import numpy as np, torch, os, sys
# from libero.libero.envs import OffScreenRenderEnv

# LIBERO_SITE = f'/usr/local/lib/python3.{sys.version_info.minor}/dist-packages/libero/libero'
# CAMERAS     = ['agentview', 'robot0_eye_in_hand']
# IMG_SIZE    = 360
# N_STATES    = 10

# WITH_SUITES = [
#     'libero_spatial_with_milk',
#     # 'libero_spatial_with_yellow_book',
#     # 'libero_goal_with_milk',
#     'libero_goal_with_yellow_book',
#     # 'libero_object_with_yellow_book',
#     # 'libero_10_with_milk',
# ]

# for suite in WITH_SUITES:
#     bddl_dir = os.path.join(LIBERO_SITE, 'bddl_files', suite)
#     save_dir = os.path.join(LIBERO_SITE, 'init_files', suite)

#     if not os.path.exists(bddl_dir):
#         print(f'SKIP (no bddl): {suite}')
#         continue

#     os.makedirs(save_dir, exist_ok=True)
#     tasks = sorted([f.replace('.bddl','') for f in os.listdir(bddl_dir) if f.endswith('.bddl')])
#     print(f'\n{suite} ({len(tasks)} tasks):')

#     for task in tasks:
#         save_path = os.path.join(save_dir, task + '.pruned_init')
#         if os.path.exists(save_path):
#             print(f'  SKIP (exists): {task[:55]}')
#             continue
#         bddl_path = os.path.join(bddl_dir, task + '.bddl')
#         try:
#             env = OffScreenRenderEnv(
#                 bddl_file_name=bddl_path,
#                 camera_names=CAMERAS,
#                 camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
#                 has_offscreen_renderer=True, use_camera_obs=True,
#                 has_renderer=False, reward_shaping=False,
#             )
#             states = []
#             for _ in range(N_STATES):
#                 env.reset()
#                 states.append(env.sim.get_state().flatten())
#             env.close()
#             states_arr = np.array(states)
#             torch.save(states_arr, save_path)
#             print(f'  {task[:55]}: {states_arr.shape}')
#         except Exception as e:
#             print(f'  ERROR {task[:45]}: {e}')

# print('\nDistractor init states complete.')

SKIP (no bddl): libero_spatial_with_milk
SKIP (no bddl): libero_goal_with_yellow_book


In [ ]:
# import shutil, os, sys

# LIBERO_SITE = f'/usr/local/lib/python3.{sys.version_info.minor}/dist-packages/libero/libero'
# INIT_SRC    = os.path.join(LIBERO_SITE, 'init_files')
# INIT_DST    = f'{RESULTS_DIR}/init_files_backup_dimensional_1'

# TARGET_SUITES = [
#     'libero_object_temp_x0.1',
#     'libero_object_temp_y0.1',
#     'libero_object_temp_x0.2',
#     'libero_object_temp_y0.2',
#     'libero_spatial_with_milk',
#     'libero_goal_with_yellow_book',
# ]

# os.makedirs(INIT_DST, exist_ok=True)

# for suite in TARGET_SUITES:
#     src = os.path.join(INIT_SRC, suite)
#     dst = os.path.join(INIT_DST, suite)
#     if not os.path.exists(src):
#         print(f'  SKIP (not found): {suite}')
#         continue
#     if os.path.exists(dst):
#         shutil.rmtree(dst)
#     shutil.copytree(src, dst)
#     n_files = len([f for f in os.listdir(dst) if f.endswith('.pruned_init')])
#     print(f'  Saved {n_files} init files: {suite}')

# print(f'\nInit files backed up to: {INIT_DST}')

  Saved 10 init files: libero_object_temp_x0.1
  Saved 10 init files: libero_object_temp_y0.1
  Saved 10 init files: libero_object_temp_x0.2
  Saved 10 init files: libero_object_temp_y0.2
  Saved 10 init files: libero_spatial_with_milk
  Saved 10 init files: libero_goal_with_yellow_book

Init files backed up to: /content/drive/MyDrive/cs159_jeff/libero_pro_results/init_files_backup_dimensional_1


# LOAD IN INIT STATES FROM DRIVE

In [ ]:
import shutil, os, sys

LIBERO_SITE = f'/usr/local/lib/python3.{sys.version_info.minor}/dist-packages/libero/libero'
INIT_SRC    = f'{RESULTS_DIR}/init_files_backup_dimensional_1'
INIT_DST    = os.path.join(LIBERO_SITE, 'init_files')

TARGET_SUITES = [
    'libero_object_temp_x0.1',
    'libero_object_temp_y0.1',
    'libero_object_temp_x0.2',
    'libero_object_temp_y0.2',
    'libero_spatial_with_milk',
    'libero_goal_with_yellow_book',
]

for suite in TARGET_SUITES:
    src = os.path.join(INIT_SRC, suite)
    dst = os.path.join(INIT_DST, suite)
    if not os.path.exists(src):
        print(f'  SKIP (not in backup): {suite}')
        continue
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    n_files = len([f for f in os.listdir(dst) if f.endswith('.pruned_init')])
    print(f'  Restored {n_files} init files: {suite}')

print(f'\nInit files restored from: {INIT_SRC}')

  Restored 10 init files: libero_object_temp_x0.1
  Restored 10 init files: libero_object_temp_y0.1
  Restored 10 init files: libero_object_temp_x0.2
  Restored 10 init files: libero_object_temp_y0.2
  Restored 10 init files: libero_spatial_with_milk
  Restored 10 init files: libero_goal_with_yellow_book

Init files restored from: /content/drive/MyDrive/cs159_jeff/libero_pro_results/init_files_backup_dimensional_1


---
## 10. Benchmark Registration & Verification
Run every session after policy is loaded.

In [ ]:
import sys, os, importlib
import libero.libero.benchmark as _bm
import libero.libero.benchmark.libero_suite_task_map as _ltm

LIBERO_SITE = f'/usr/local/lib/python3.{sys.version_info.minor}/dist-packages/libero/libero'

# Force reload task map
importlib.reload(_ltm)
importlib.reload(_bm)

libero_task_map = _ltm.libero_task_map

TEMP_STRENGTHS = (
    [f'libero_object_temp_x{v}' for v in ['0.1','0.2','0.3','0.4','0.5']] +
    [f'libero_object_temp_y{v}' for v in ['0.1','0.2','0.3','0.4','0.5']]
)

# ── Monkey-patch get_benchmark_dict to include position perturbation suites ───
_orig_get_bm_dict = _bm.get_benchmark_dict

def _patched_get_benchmark_dict():
    result = dict(_orig_get_bm_dict())

    for suite in TEMP_STRENGTHS:
        if suite not in result and \
           suite in _bm.task_maps and \
           _bm.task_maps[suite]:

            def make_cls(s):
                class _PositionPerturbSuite:
                    def __init__(self):
                        self._tasks = list(_bm.task_maps[s].values())
                        self.n_tasks = len(self._tasks)
                    def get_task(self, i):
                        return self._tasks[i]
                    def get_task_names(self):
                        return list(_bm.task_maps[s].keys())
                    def get_task_init_states(self, i):
                        import torch, os, sys
                        task = self._tasks[i]
                        init_dir = os.path.join(
                            f'/usr/local/lib/python3.{sys.version_info.minor}'
                            f'/dist-packages/libero/libero/init_files',
                            task.problem_folder)
                        init_file = os.path.join(
                            init_dir,
                            task.bddl_file.replace('.bddl', '.pruned_init'))
                        return torch.load(init_file)
                return _PositionPerturbSuite

            result[suite] = make_cls(suite)

    return result

_bm.get_benchmark_dict = _patched_get_benchmark_dict

patched_dict = _bm.get_benchmark_dict()
found = [k for k in patched_dict if 'temp_x' in k or 'temp_y' in k]
print(f'Position perturbation suites now in benchmark_dict: {found}')

# ── Register object aliases ───────────────────────────────────────────────────
from libero.libero.envs.objects import OBJECTS_DICT

ALIASES = {
    'black_bowl':              'akita_black_bowl',
    'yellow_plate':            'plate',
    'bigger_akita_black_bowl': 'akita_black_bowl',
    'brown_rack':              'wine_rack',
    'red_cream_cheese':        'cream_cheese',
    'white_bottle':            'wine_bottle',
    'yellow_cabinet':          'wooden_cabinet',
    'yellow_stove':            'stove',
}
for alias, base in ALIASES.items():
    if alias not in OBJECTS_DICT and base in OBJECTS_DICT:
        OBJECTS_DICT[alias] = OBJECTS_DICT[base]

print(f'OBJECTS_DICT: {len(OBJECTS_DICT)} entries')

# ── Build MAX_STEPS_MAP and verify all target suites ─────────────────────────
MAX_STEPS_MAP = {}
bm_dict = _bm.get_benchmark_dict()

TARGET_SUITES = [
    # Position perturbation (known to be in 20-60% SR range)
    'libero_object_temp_x0.1',
    'libero_object_temp_y0.1',
    'libero_object_temp_y0.2',
    'libero_object_temp_x0.2',
    # Distractor (asset-verified)
    'libero_spatial_with_milk',
    # 'libero_spatial_with_yellow_book',
    # 'libero_goal_with_milk',
    'libero_goal_with_yellow_book',
    # 'libero_object_with_yellow_book',
    # 'libero_10_with_milk',
]

MAX_STEPS_DEFAULTS = {
    'libero_10': 520, 'libero_goal': 300, 'libero_object': 280,
    'libero_spatial': 300, 'libero_object_temp': 280,
}

print('\nSuite verification:')
print(f'{"Suite":<45} {"Tasks":>5}  {"Max steps":>10}  Status')
print('-' * 75)
for suite in TARGET_SUITES:
    if suite in bm_dict:
        n = bm_dict[suite]().n_tasks
        ms = 280
        for prefix, steps in MAX_STEPS_DEFAULTS.items():
            if suite.startswith(prefix):
                ms = steps
                break
        MAX_STEPS_MAP[suite] = ms
        print(f'{suite:<45} {n:>5}  {ms:>10}  ✓')
    else:
        print(f'{suite:<45} {"":>5}  {"":>10}  ✗ NOT IN BENCHMARK DICT')

print(f'\n{len(MAX_STEPS_MAP)}/{len(TARGET_SUITES)} suites ready.')

Position perturbation suites now in benchmark_dict: ['libero_object_temp_x0.1', 'libero_object_temp_x0.2', 'libero_object_temp_x0.3', 'libero_object_temp_x0.4', 'libero_object_temp_x0.5', 'libero_object_temp_y0.1', 'libero_object_temp_y0.2', 'libero_object_temp_y0.3', 'libero_object_temp_y0.4', 'libero_object_temp_y0.5']
OBJECTS_DICT: 62 entries

Suite verification:
Suite                                         Tasks   Max steps  Status
---------------------------------------------------------------------------
libero_object_temp_x0.1                          10         280  ✓
libero_object_temp_y0.1                          10         280  ✓
libero_object_temp_y0.2                          10         280  ✓
libero_object_temp_x0.2                          10         280  ✓
[info] Using default task order for benchmark 'libero_spatial_with_milk' (10 tasks).
libero_spatial_with_milk                         10         300  ✓
[info] Using default task order for benchmark 'libero_goal_with

---
## 11. P&P Implementation
Monkey-patches `policy.model.sample_actions` to inject the predict-and-perturb loop.
Run every session after policy is loaded.

Fix: Changes vs. your version are marked # FIX: / # NEW:. Everything else is identical so it stays compatible with write_episode_to_db, PNP_RECORDER, and the DB schema.

In [ ]:
import torch, numpy as np, types, uuid, json as _json, math
from dataclasses import dataclass
from typing import Optional, Sequence
from lerobot.policies.pi05.modeling_pi05 import make_att_2d_masks


@dataclass
class PnPConfig:
    enabled:              bool                    = False
    step_indices:         Optional[Sequence[int]] = (1,)
    time_min:             Optional[float]         = None
    num_iterations:       int                     = 3
    mode:                 str                     = 'both'
    action_dim:           int                     = 7
    record_per_iteration: bool                    = False
    # NEW:
    perturb_seed:         int                     = 12345   # dedicated perturb-noise seed
    compute_multimodal:   bool                    = True    # log BC / PC1 modality stats

    def step_selected(self, step: int, s: float) -> bool:
        if not self.enabled: return False
        if self.time_min is not None: return s >= self.time_min
        return self.step_indices is not None and step in tuple(self.step_indices)

    @property
    def do_refine(self) -> bool:
        return self.mode in ('refine', 'both')

    @property
    def step_indices_json(self) -> Optional[str]:
        return _json.dumps(list(self.step_indices)) if self.step_indices is not None else None


# NEW: per-device generators for perturbation noise. These are SEPARATE from the global
# RNG, so running the predict-perturb loop never advances the stream that samples chunk
# noise. -> uncertainty mode is a true no-op; refinement reflects only state replacement.
_PNP_PERTURB_GENS = {}
def _pnp_perturb_gen(device):
    key = str(device)
    g = _PNP_PERTURB_GENS.get(key)
    if g is None:
        g = torch.Generator(device=device)
        g.manual_seed(PNP_CONFIG.perturb_seed)
        _PNP_PERTURB_GENS[key] = g
    return g

def pnp_reset_perturb_gens(seed=None):
    """Call at the start of each episode for reproducible perturbations (optional)."""
    if seed is not None:
        PNP_CONFIG.perturb_seed = int(seed)
    _PNP_PERTURB_GENS.clear()


# ---- multimodality helpers (numpy, cheap; run on the K stacked predictions) ----
def _bc_1d(x):
    """Sarle's bimodality coefficient on a 1-D array of K samples. >0.555 ~ bimodal."""
    n = x.shape[0]
    if n < 4: return float('nan')
    d = x - x.mean()
    s = np.sqrt((d**2).mean()) + 1e-12
    g = (d**3).mean() / s**3                       # skewness
    k = (d**4).mean() / s**4 - 3.0                 # excess kurtosis
    return float((g**2 + 1.0) / (k + 3.0*(n-1)**2/((n-2)*(n-3))))

def _multimodal_stats(A0):
    """A0: (K, chunk, adim) numpy. Returns per-dim BC, PC1 variance fraction, BC of PC1."""
    K = A0.shape[0]
    # per-dim bimodality coefficient, computed per chunk-timestep then averaged
    d  = A0 - A0.mean(axis=0, keepdims=True)
    s2 = (d**2).mean(axis=0); sd = np.sqrt(s2) + 1e-12
    g  = (d**3).mean(axis=0) / sd**3               # (chunk, adim) skew
    k  = (d**4).mean(axis=0) / sd**4 - 3.0         # (chunk, adim) excess kurt
    bc = (g**2 + 1.0) / (k + 3.0*(K-1)**2/((K-2)*(K-3)))
    bc_vec = np.nanmean(bc, axis=0)                # (adim,)
    # global modality: top PC of the flattened predictions
    F = A0.reshape(K, -1).astype(np.float64)
    F = F - F.mean(axis=0, keepdims=True)
    try:
        U, S, _ = np.linalg.svd(F, full_matrices=False)
        pc1_frac = float(S[0]**2 / (S**2).sum())
        bc_pc1   = _bc_1d(U[:, 0] * S[0])          # modality along dominant direction
    except np.linalg.LinAlgError:
        pc1_frac, bc_pc1 = float('nan'), float('nan')
    return bc_vec, pc1_frac, bc_pc1


def _pnp_refine_at_step(x_t, s, vfield, cfg):
    adim   = cfg.action_dim
    x_acc  = x_t
    a_hats = []
    gen    = _pnp_perturb_gen(x_acc.device)        # NEW
    for _ in range(cfg.num_iterations):
        v     = vfield(x_acc)
        a_hat = x_acc - s * v
        a_hats.append(a_hat[..., :adim])
        # FIX: draw from the dedicated generator, NOT torch.randn_like (which uses global RNG)
        eps   = torch.empty_like(x_acc).normal_(generator=gen)
        x_acc = (1.0 - s) * a_hat + s * eps

    A = torch.stack(a_hats, dim=0)                 # (K, B, chunk, adim)
    if A.shape[0] >= 2:
        u_consecutive = (A[1:] - A[:-1]).abs().mean(dim=0)   # (B, chunk, adim)
        a_std         = A.std(dim=0)
    else:
        u_consecutive = torch.zeros_like(A[0]); a_std = torch.zeros_like(A[0])

    u_vec_per_dim = u_consecutive.squeeze(0).mean(dim=0)     # (adim,)
    a_mean_vec    = A.mean(dim=0).squeeze(0).mean(dim=0)
    a_std_vec     = a_std.squeeze(0).mean(dim=0)             # NEW: per-dim spread across K

    rec = {
        's':          float(s),
        'u_mean':     float(u_consecutive.mean()),
        'u_max':      float(u_consecutive.max()),
        'u_vec':      u_vec_per_dim.detach().float().cpu().numpy().tolist(),
        'a_mean_vec': a_mean_vec.detach().float().cpu().numpy().tolist(),
        'a_std_vec':  a_std_vec.detach().float().cpu().numpy().tolist(),   # NEW
        'a_std_mean': float(a_std.mean()),
    }

    # NEW: online multimodality stats (needs K>=4; set num_iterations>=15 for a usable estimate)
    if cfg.compute_multimodal and A.shape[0] >= 4:
        A0 = A.detach().float().cpu().numpy()[:, 0]          # (K, chunk, adim)
        bc_vec, pc1_frac, bc_pc1 = _multimodal_stats(A0)
        rec['bc_vec']     = bc_vec.tolist()
        rec['mm_pc1_frac'] = float(pc1_frac)
        rec['mm_bc_pc1']   = float(bc_pc1)

    if cfg.record_per_iteration:
        rec['a_hats'] = A.detach().float().cpu().numpy()

    return (x_acc if cfg.do_refine else x_t), rec


@torch.no_grad()
def _sample_actions_pnp(self, images, img_masks, tokens, masks,
                          noise=None, num_steps=None, **kwargs):
    cfg = PNP_CONFIG
    if (not cfg.enabled) or self._rtc_enabled():
        return self._orig_sample_actions(
            images, img_masks, tokens, masks, noise=noise, num_steps=num_steps, **kwargs)
    if num_steps is None:
        num_steps = self.config.num_inference_steps
    bsize  = tokens.shape[0]; device = tokens.device
    if noise is None:
        noise = self.sample_noise(
            (bsize, self.config.chunk_size, self.config.max_action_dim), device)
    prefix_embs, prefix_pad_masks, prefix_att_masks = \
        self.embed_prefix(images, img_masks, tokens, masks)
    prefix_att_2d_masks    = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
    prefix_position_ids    = torch.cumsum(prefix_pad_masks, dim=1) - 1
    prefix_att_2d_masks_4d = self._prepare_attention_masks_4d(prefix_att_2d_masks)
    self.paligemma_with_expert.paligemma.model.language_model.config._attn_implementation = 'eager'
    _, past_key_values = self.paligemma_with_expert.forward(
        attention_mask=prefix_att_2d_masks_4d, position_ids=prefix_position_ids,
        past_key_values=None, inputs_embeds=[prefix_embs, None], use_cache=True)
    dt = -1.0 / num_steps; x_t = noise
    chunk_rec = {'num_steps': num_steps, 'steps': []}
    for step in range(num_steps):
        s = 1.0 + step * dt
        time_tensor = torch.tensor(s, dtype=torch.float32, device=device).expand(bsize)
        def vfield(inp, _ts=time_tensor):
            return self.denoise_step(prefix_pad_masks=prefix_pad_masks,
                past_key_values=past_key_values, x_t=inp, timestep=_ts)
        if cfg.step_selected(step, s):
            x_t, rec = _pnp_refine_at_step(x_t, s, vfield, cfg)
            rec['step'] = step; chunk_rec['steps'].append(rec)
        x_t = x_t + dt * vfield(x_t)
    PNP_RECORDER.log_chunk(chunk_rec)
    return x_t


class PnPRecorder:
    def __init__(self):
        self.reset()
    def reset(self):
        self.episodes   = []
        self._cur       = None
        self._chunk_idx = 0
    def new_episode(self, meta: dict = None):
        self._cur = {
            'rollout_id':     str(uuid.uuid4()),
            'meta':           dict(meta or {}),
            'chunks':         [],
            'success':        None,
            'n_steps':        None,
            'elapsed_s':      None,
            'u_mean_episode': None,
            'u_max_episode':  None,
        }
        self._chunk_idx = 0
    def log_chunk(self, chunk_rec: dict):
        if self._cur is None: return
        chunk_rec = dict(chunk_rec)
        chunk_rec['chunk_idx'] = self._chunk_idx
        self._cur['chunks'].append(chunk_rec)
        self._chunk_idx += 1
    def close_episode(self, success: bool, n_steps: int, elapsed_s: float = 0.0):
        if self._cur is None: return
        self._cur['success']   = bool(success)
        self._cur['n_steps']   = int(n_steps)
        self._cur['elapsed_s'] = float(elapsed_s)
        all_u = [st['u_mean'] for c in self._cur['chunks'] for st in c['steps']]
        if all_u:
            self._cur['u_mean_episode'] = float(np.mean(all_u))
            self._cur['u_max_episode']  = float(np.max(all_u))
        self.episodes.append(self._cur)
        self._cur = None
    @property
    def current_rollout_id(self):
        return self._cur['rollout_id'] if self._cur else None


PNP_CONFIG   = PnPConfig()
PNP_RECORDER = globals().get('PNP_RECORDER', None) or PnPRecorder()   # reuse if already defined

try:
    from lerobot.constants import ACTION
    PNP_CONFIG.action_dim = policy.config.output_features[ACTION].shape[0]
except Exception:
    PNP_CONFIG.action_dim = 7
if not hasattr(policy.model, '_orig_sample_actions'):
    policy.model._orig_sample_actions = policy.model.sample_actions
policy.model.sample_actions = types.MethodType(_sample_actions_pnp, policy.model)
PNP_CONFIG.enabled = False
print(f'P&P (fixed) ready. action_dim={PNP_CONFIG.action_dim}  '
      f'inference_steps={policy.config.num_inference_steps}')
print('Perturb noise now uses a dedicated generator -> uncertainty mode is a true no-op.')


P&P (fixed) ready. action_dim=7  inference_steps=10
Perturb noise now uses a dedicated generator -> uncertainty mode is a true no-op.


# Sanity check the no-op fix

Run after Sampler definition cell. Confirms that with mode='uncertainty' the executed action chunk is bit-identical to baseline for the same noise (the property the old code violated).

In [ ]:
import torch, numpy as np

def _dummy_inputs():
    # Build one real batch from a single env step so shapes/conditioning are valid.
    bm_dict = _bm.get_benchmark_dict(); ts = bm_dict['libero_object_temp_x0.1']()
    task = ts.get_task(0); init_states = ts.get_task_init_states(0)
    bddl = os.path.join(get_libero_path('bddl_files'), task.problem_folder, task.bddl_file)
    env = OffScreenRenderEnv(bddl_file_name=bddl, camera_names=CAMERAS,
        camera_heights=IMG_SIZE, camera_widths=IMG_SIZE, has_offscreen_renderer=True,
        use_camera_obs=True, has_renderer=False, reward_shaping=False)
    env.reset(); policy.reset(); obs = env.set_init_state(init_states[0])
    for _ in range(NUM_STEPS_WAIT): obs,_,_,_ = env.step(LIBERO_DUMMY_ACTION)
    batch = preprocess(obs_to_policy(obs, task.language, device)); env.close()
    return batch

batch = _dummy_inputs()

def _chunk(seed):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    policy.reset()
    with torch.no_grad(): a = policy.select_action(batch)
    return postprocess(a) if not isinstance(a, torch.Tensor) else a.squeeze(0).cpu().numpy()

PNP_CONFIG.enabled=False
base = _chunk(0)
PNP_CONFIG.enabled=True; PNP_CONFIG.mode='uncertainty'
PNP_CONFIG.step_indices=(2,3); PNP_CONFIG.num_iterations=15
unc = _chunk(0)
PNP_CONFIG.enabled=False

max_abs = float(np.max(np.abs(np.asarray(base) - np.asarray(unc))))
print(f'max |baseline - uncertainty| over first action = {max_abs:.2e}')
print('PASS (true no-op)' if max_abs < 1e-5 else 'STILL DIVERGING — investigate noise path')


Local assets not found. Downloading from HuggingFace Hub...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 586 files:   0%|          | 0/586 [00:00<?, ?it/s]

Assets downloaded successfully to /root/.cache/libero/assets
max |baseline - uncertainty| over first action = 0.00e+00
PASS (true no-op)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---
## 12. Database Setup & Eval Helpers
Run every session.

Fix: Adds columns for the new per-step signals. Idempotent. Swap your write_episode_to_db for this one (it is a superset — old columns unchanged).

In [ ]:
import sqlite3, os, time as _time, hashlib, json as _json, numpy as np, torch

# DB_PATH = f'{RESULTS_DIR}/rollouts_jeff_2_k7.db'
con     = sqlite3.connect(DB_PATH, check_same_thread=False)

con.executescript("""
    CREATE TABLE IF NOT EXISTS rollouts (
        rollout_id          TEXT PRIMARY KEY,
        suite               TEXT,
        task_idx            INTEGER,
        episode_idx         INTEGER,
        init_state_hash     TEXT,
        success             INTEGER,
        n_steps             INTEGER,
        elapsed_s           REAL,
        pnp_enabled         INTEGER,
        pnp_step_indices    TEXT,
        pnp_mode            TEXT,
        pnp_num_iterations  INTEGER,
        u_mean_episode      REAL,
        u_max_episode       REAL,
        timestamp           TEXT
    );

    CREATE TABLE IF NOT EXISTS pnp_euler_steps (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        rollout_id  TEXT,
        chunk_idx   INTEGER,
        euler_step  INTEGER,
        u_mean      REAL,
        u_max       REAL,
        s           REAL
    );

    CREATE TABLE IF NOT EXISTS pnp_action_vectors (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        rollout_id   TEXT,
        chunk_idx    INTEGER,
        euler_step   INTEGER,
        s            REAL,
        u_vec        TEXT,   -- JSON list of 7 floats: per-dim uncertainty
        a_mean_vec   TEXT    -- JSON list of 7 floats: mean predicted clean action per dim
    );

""")
con.commit()

existing = con.execute('SELECT COUNT(*) FROM rollouts').fetchone()[0]
print(f'DB ready: {DB_PATH}  ({existing} existing rollouts)')


def _init_state_hash(init_state_arr):
    return hashlib.md5(np.asarray(init_state_arr).tobytes()).hexdigest()[:16]


import json as _json, time as _time

def _ensure_cols(con, table, cols):
    have = {r[1] for r in con.execute(f'PRAGMA table_info({table})').fetchall()}
    for name, decl in cols:
        if name not in have:
            con.execute(f'ALTER TABLE {table} ADD COLUMN {name} {decl}')
    con.commit()

_ensure_cols(con, 'pnp_action_vectors', [
    ('a_std_vec', 'TEXT'), ('bc_vec', 'TEXT'),
    ('mm_pc1_frac', 'REAL'), ('mm_bc_pc1', 'REAL')])
_ensure_cols(con, 'rollouts', [('mm_bc_pc1_episode', 'REAL')])

def write_episode_to_db(ep, suite, task_idx, episode_idx, init_state_hash, elapsed_s):
    rid = ep['rollout_id']
    # per-episode aggregate modality = mean PC1 bimodality across logged steps
    bcs = [st.get('mm_bc_pc1') for c in ep.get('chunks', []) for st in c.get('steps', [])
           if st.get('mm_bc_pc1') is not None and not (st.get('mm_bc_pc1') != st.get('mm_bc_pc1'))]
    mm_bc_ep = float(np.mean(bcs)) if bcs else None
    con.execute("""INSERT OR REPLACE INTO rollouts
        (rollout_id, suite, task_idx, episode_idx, init_state_hash, success, n_steps,
         elapsed_s, pnp_enabled, pnp_step_indices, pnp_mode, pnp_num_iterations,
         u_mean_episode, u_max_episode, mm_bc_pc1_episode, timestamp)
        VALUES (?,?,?,?,?, ?,?,?, ?,?,?,?, ?,?,?, ?)""", (
        rid, suite, task_idx, episode_idx, init_state_hash, int(ep['success']),
        ep['n_steps'], elapsed_s, int(PNP_CONFIG.enabled), PNP_CONFIG.step_indices_json,
        PNP_CONFIG.mode if PNP_CONFIG.enabled else None,
        PNP_CONFIG.num_iterations if PNP_CONFIG.enabled else None,
        ep.get('u_mean_episode'), ep.get('u_max_episode'), mm_bc_ep,
        _time.strftime('%Y-%m-%dT%H:%M:%S')))
    for chunk in ep.get('chunks', []):
        for st in chunk.get('steps', []):
            con.execute("""INSERT INTO pnp_euler_steps
                (rollout_id, chunk_idx, euler_step, u_mean, u_max, s) VALUES (?,?,?,?,?,?)""",
                (rid, chunk['chunk_idx'], st['step'], st['u_mean'], st['u_max'], st['s']))
            if 'u_vec' in st:
                con.execute("""INSERT INTO pnp_action_vectors
                    (rollout_id, chunk_idx, euler_step, s, u_vec, a_mean_vec,
                     a_std_vec, bc_vec, mm_pc1_frac, mm_bc_pc1)
                    VALUES (?,?,?,?,?,?,?,?,?,?)""", (
                    rid, chunk['chunk_idx'], st['step'], st['s'],
                    _json.dumps(st['u_vec']), _json.dumps(st.get('a_mean_vec', [])),
                    _json.dumps(st.get('a_std_vec', [])), _json.dumps(st.get('bc_vec', [])),
                    st.get('mm_pc1_frac'), st.get('mm_bc_pc1')))
    con.commit()

print('Schema extended; write_episode_to_db now logs a_std_vec / bc_vec / mm_* fields.')


# ── Eval helpers ─────────────────────────────────────────────────────────────
import torchvision.transforms.functional as TVF

CAMERAS           = ['agentview', 'robot0_eye_in_hand']
IMG_SIZE          = 360
LIBERO_DUMMY_ACTION = [0.0] * 6 + [-1.0]
NUM_STEPS_WAIT    = 10


def _quat2axisangle(quat):
    import numpy as np
    q = np.array(quat, dtype=float)
    if q[3] < 0:
        q = -q
    denom = np.sqrt(1.0 - q[3]**2)
    if denom < 1e-8:
        return np.zeros(3)
    return (q[:3] / denom) * 2.0 * np.arccos(q[3])


def run_episode(env, init_state, policy, task_desc, max_steps, device):
    env.reset()
    policy.reset()
    obs = env.set_init_state(init_state)
    for _ in range(NUM_STEPS_WAIT):
        obs, _, _, _ = env.step(LIBERO_DUMMY_ACTION)
    t0 = _time.time()
    for step in range(max_steps):
        raw_obs = obs_to_policy(obs, task_desc, device)
        batch   = preprocess(raw_obs)
        with torch.no_grad():
            action = policy.select_action(batch)
        action = postprocess(action)
        obs, _, done, _ = env.step(action)
        if env.check_success():
            return True, step + 1, _time.time() - t0
        if done:
            break
    return False, step + 1, _time.time() - t0


print('DB and eval helpers ready.')

DB ready: /content/drive/MyDrive/cs159_jeff/libero_pro_results/rollouts_jeff_2_k7.db  (0 existing rollouts)
Schema extended; write_episode_to_db now logs a_std_vec / bc_vec / mm_* fields.
DB and eval helpers ready.


Samples N independent action chunks at a fixed observation (P&P OFF), then measures their variance. This is the policy's own output uncertainty — the natural baseline for "is the free P&P signal as good as the expensive one?" It does NOT rank by P&P U (unlike the existing multi_sample_select).

One integration point to verify (_sample_one_chunk): it must return the full predicted action chunk (T, adim) for one forward pass. The template below uses select_action's underlying chunk; if your lerobot version exposes policy.predict_action_chunk or similar, wire it here. Global RNG is snapshotted/restored so this measurement never perturbs the primary rollout.

In [ ]:
import torch, numpy as np, json as _json

con.executescript("""
CREATE TABLE IF NOT EXISTS baseline_uncertainty (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    rollout_id TEXT, chunk_idx INTEGER, n_samples INTEGER,
    ms_var_vec TEXT,      -- JSON: per-dim variance across N samples (mean over chunk)
    ms_pair_l2 REAL,      -- mean pairwise L2 between the N chunks
    init_state_hash TEXT, suite TEXT
);""")
con.commit()
_ensure_cols(con, 'baseline_uncertainty', [('init_state_hash','TEXT'), ('suite','TEXT')])

def _sample_one_chunk(policy, batch):
    """RETURN the full predicted action chunk as np.ndarray (T, adim) for ONE forward pass.
    VERIFY this against your lerobot API. Common options:
      a = policy.predict_action_chunk(batch)            # if available -> (T, adim)
      a = policy.model.sample_actions(...)              # lower level
    Fallback below dequeues one chunk via select_action internals."""
    policy.reset()
    with torch.no_grad():
        if hasattr(policy, 'predict_action_chunk'):
            a = policy.predict_action_chunk(batch)
        else:
            a = policy.select_action(batch)             # may return only first action
    a = a.detach().cpu().numpy() if isinstance(a, torch.Tensor) else np.asarray(a)
    return a.squeeze()

def measure_multi_sample(policy, batch, n_samples=8, base_seed=0):
    """N independent chunks at the SAME obs; returns (per-dim var vec, mean pairwise L2).
    Snapshots/restores global RNG so the main rollout is unaffected."""
    cpu_state = torch.get_rng_state()
    cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    saved = PNP_CONFIG.enabled; PNP_CONFIG.enabled = False
    chunks = []
    try:
        for i in range(n_samples):
            torch.manual_seed(base_seed + i); torch.cuda.manual_seed(base_seed + i)
            chunks.append(np.atleast_2d(_sample_one_chunk(policy, batch)))
    finally:
        PNP_CONFIG.enabled = saved
        torch.set_rng_state(cpu_state)
        if cuda_state is not None: torch.cuda.set_rng_state_all(cuda_state)
    A = np.stack(chunks, axis=0)                          # (N, T, adim)  (T may be 1)
    var_vec = A.var(axis=0).mean(axis=0)                  # (adim,)
    # mean pairwise L2 over flattened chunks
    F = A.reshape(A.shape[0], -1)
    d = [np.linalg.norm(F[i] - F[j]) for i in range(len(F)) for j in range(i+1, len(F))]
    return var_vec, float(np.mean(d) if d else 0.0)

# Example call inside your eval loop, once per chunk boundary, on the SAME `batch`
# that produced the executed chunk (run this in mode='uncertainty' rollouts):
#   vv, pl2 = measure_multi_sample(policy, batch, n_samples=8,
#                                  base_seed=hash(rid) % (2**31))
#   con.execute("INSERT INTO baseline_uncertainty (rollout_id, chunk_idx, n_samples,
#                ms_var_vec, ms_pair_l2) VALUES (?,?,?,?,?)",
#               (rid, chunk_idx, 8, _json.dumps(vv.tolist()), pl2)); con.commit()
print('measure_multi_sample ready. COST: ~N x inference per probed chunk. Verify _sample_one_chunk.')


measure_multi_sample ready. COST: ~N x inference per probed chunk. Verify _sample_one_chunk.


In [ ]:
# ── Smoke test: confirm measure_multi_sample / _sample_one_chunk shapes ──
import numpy as np, torch
try:
    _b = batch                 # left in scope by the no-op sanity-check cell
except NameError:
    _b = _dummy_inputs()
one = np.atleast_2d(_sample_one_chunk(policy, _b))
print(f'_sample_one_chunk -> shape {one.shape}  '
      f'({"full chunk (T, adim)" if one.shape[0] > 1 else "single action — select_action path"})')
vv, pl2 = measure_multi_sample(policy, _b, n_samples=4, base_seed=0)
print(f'ms_var_vec shape = {vv.shape}   (expect ({PNP_CONFIG.action_dim},))')
print(f'ms_var_vec       = {np.round(vv, 5)}')
print(f'ms_pair_l2       = {pl2:.4f}')
ok = (vv.shape == (PNP_CONFIG.action_dim,)) and np.all(np.isfinite(vv)) and np.isfinite(pl2)
print('PASS — shapes/values sane' if ok else 'CHECK — verify _sample_one_chunk')


_sample_one_chunk -> shape (50, 7)  (full chunk (T, adim))
ms_var_vec shape = (7,)   (expect (7,))
ms_var_vec       = [5.02e-03 3.57e-03 1.47e-03 9.02e-03 5.62e-03 1.66e-03 3.00e-05]
ms_pair_l2       = 1.8675
PASS — shapes/values sane


---
## 13. Evaluation Configuration
Edit this cell to select suites and episode count, then run the eval loop below.

In [ ]:
# ── Eval configuration — edit as needed ──────────────────────────────────────

SUITES = [
    # Position perturbation suites (known SR range: 27-60%)
    'libero_object_temp_x0.1',   # ~53% baseline
    'libero_object_temp_y0.1',   # ~60% baseline
    'libero_object_temp_y0.2',   # ~27% baseline
    'libero_object_temp_x0.2',   # ~0% baseline
    # Distractor suites (untested — run first to confirm SR is in 20-70% range)
    'libero_spatial_with_milk',
    # 'libero_spatial_with_yellow_book',
    # 'libero_goal_with_milk',
    'libero_goal_with_yellow_book',
    # 'libero_object_with_yellow_book',
    # 'libero_10_with_milk',
]

N_EPISODES = 10   # episodes per task; use 1 for smoke test
VERBOSE    = True

print(f'Suites:    {SUITES}')
print(f'Episodes:  {N_EPISODES}/task')
print(f'P&P mode:  {"ENABLED — " + PNP_CONFIG.mode if PNP_CONFIG.enabled else "disabled (baseline)"}')
print(f'Max steps: { {s: MAX_STEPS_MAP.get(s, 300) for s in SUITES} }')

# The following were used (and kept) for a multi-sample variance experiment. Currently unused
# # ── Multi-sample variance baseline: config ──
# MS_BASELINE_ENABLED = False     # log multi-sample variance during uncertainty-pass rollouts
# MS_N_SAMPLES        = 5         # samples per measurement (~N full chunk inferences each)

# # the following are only used if MS_BASELINE_ENABLED = False (there is a cell below that can run MS standalone)
# MS_N_EPISODES       = 10        # episodes/task for the standalone baseline pass
# MS_SUITES           = list(SUITES)
# print(f'MS baseline: {"ON" if MS_BASELINE_ENABLED else "off"}  (N={MS_N_SAMPLES})')


Suites:    ['libero_object_temp_x0.1', 'libero_object_temp_y0.1', 'libero_object_temp_y0.2', 'libero_object_temp_x0.2', 'libero_spatial_with_milk', 'libero_goal_with_yellow_book']
Episodes:  10/task
P&P mode:  ENABLED — uncertainty
Max steps: {'libero_object_temp_x0.1': 280, 'libero_object_temp_y0.1': 280, 'libero_object_temp_y0.2': 280, 'libero_object_temp_x0.2': 280, 'libero_spatial_with_milk': 300, 'libero_goal_with_yellow_book': 300}
MS baseline: off  (N=5)


In case needed to load distractor assets (sometimes it's buggy)

In [ ]:
import os, shutil, sys
LIBERO_SITE = f'/usr/local/lib/python3.{sys.version_info.minor}/dist-packages/libero/libero'
REPO_BDDL   = '/content/LIBERO-PRO/libero/libero/bddl_files'
SITE_BDDL   = os.path.join(LIBERO_SITE, 'bddl_files')
NEED = ['libero_spatial_with_milk', 'libero_goal_with_yellow_book']

print('repo bddl dir exists:', os.path.isdir(REPO_BDDL))
repo_suites = sorted(os.listdir(REPO_BDDL)) if os.path.isdir(REPO_BDDL) else []
print('with_/distractor folders in repo:',
      [d for d in repo_suites if any(k in d for k in ('with_', 'milk', 'book'))])

for s in NEED:
    src, dst = os.path.join(REPO_BDDL, s), os.path.join(SITE_BDDL, s)
    if os.path.isdir(src):
        if os.path.isdir(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print('copied bddl:', s, f'({len(os.listdir(dst))} files)')
    else:
        print('NOT in repo:', s)

# the distractor suites also need init states at rollout time — check they're present
for s in NEED:
    ip = os.path.join(LIBERO_SITE, 'init_files', s)
    n = len([f for f in os.listdir(ip) if f.endswith('.pruned_init')]) if os.path.isdir(ip) else 0
    print('init states for', s, ':', n if n else 'MISSING')

if any(not os.path.isdir(os.path.join(SITE_BDDL, s)) for s in NEED):
    print('\nFULL repo suite list (so we can find the right names):')
    for d in repo_suites: print(' ', d)

repo bddl dir exists: True
with_/distractor folders in repo: ['libero_10_with_blue_stick', 'libero_10_with_diffpos_stick', 'libero_10_with_milk', 'libero_10_with_mug', 'libero_10_with_red_box', 'libero_10_with_red_stick', 'libero_goal_with_blue_stick', 'libero_goal_with_diffpos_stick', 'libero_goal_with_green_mug', 'libero_goal_with_milk', 'libero_goal_with_mug', 'libero_goal_with_red_box', 'libero_goal_with_red_stick', 'libero_goal_with_rotated_stick', 'libero_goal_with_yellow_book', 'libero_object_with_blue_stick', 'libero_object_with_diffpos_stick', 'libero_object_with_mug', 'libero_object_with_red_box', 'libero_object_with_red_stick', 'libero_object_with_trigger', 'libero_object_with_trigger_new', 'libero_object_with_yellow_book', 'libero_spatial_with_blue_stick', 'libero_spatial_with_diffpos_stick', 'libero_spatial_with_green_mug', 'libero_spatial_with_milk', 'libero_spatial_with_mug', 'libero_spatial_with_red_box', 'libero_spatial_with_red_stick', 'libero_spatial_with_yellow_book

---
## 14. Evaluation Loop
Run this cell for each phase (baseline, uncertainty, refinement).
Results are appended to `rollouts.db` and can be re-analyzed at any time.

**Estimated runtime:** ~20 s/episode (baseline) · ~40 s/episode (P&P)
- 6 suites × 10 tasks × 10 ep × 3 phases ≈ 1800 episodes ≈ 10–14 hours total on A100
- Reduce `N_EPISODES = 5` for a ~5–7 hour run

In [ ]:
import hashlib
from tqdm.notebook import tqdm

def _hash_init_state(init_state) -> str:
    arr = np.asarray(init_state)
    return hashlib.md5(arr.tobytes()).hexdigest()[:16]


def run_episode_pnp(env, init_state, policy, task_desc,
                    max_steps, device, meta: dict = None):
    env.reset(); policy.reset()

    # Deterministic per-episode seed (keyed on init state) so baseline / uncertainty /
    # refinement passes start from the SAME noise -> valid cross-phase pairing.
    _seed = int(_hash_init_state(init_state), 16) % (2**31)
    torch.manual_seed(_seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(_seed)

    obs = env.set_init_state(init_state)
    for _ in range(NUM_STEPS_WAIT):
        obs, _, _, _ = env.step(LIBERO_DUMMY_ACTION)

    PNP_RECORDER.new_episode(meta)

    # Optional multi-sample variance baseline (toggle in the config cell). Measured once
    # on the first obs; restores global RNG so it does not perturb this rollout.
    if globals().get('MS_BASELINE_ENABLED', False) and PNP_CONFIG.enabled and PNP_CONFIG.mode == 'uncertainty':
        _b0  = preprocess(obs_to_policy(obs, task_desc, device))
        _rid = PNP_RECORDER._cur['rollout_id']
        _h   = _hash_init_state(init_state)
        _msv, _msl2 = measure_multi_sample(policy, _b0, n_samples=MS_N_SAMPLES,
                                           base_seed=int(_h, 16) % (2**31))
        con.execute("INSERT INTO baseline_uncertainty (rollout_id, chunk_idx, n_samples, "
                    "ms_var_vec, ms_pair_l2, init_state_hash, suite) VALUES (?,?,?,?,?,?,?)",
                    (_rid, 0, MS_N_SAMPLES, _json.dumps(_msv.tolist()), _msl2, _h,
                     meta.get('suite') if meta else None))
        con.commit()

    t0 = time.time(); success = False; step = 0

    for step in range(max_steps):
        raw_obs = obs_to_policy(obs, task_desc, device)
        batch   = preprocess(raw_obs)
        with torch.no_grad():
            action = policy.select_action(batch)
        action = postprocess(action)
        if isinstance(action, torch.Tensor):
            action = action.squeeze(0).cpu().numpy()
        obs, _, done, _ = env.step(action)
        if env.check_success():
            success = True; break
        if done:
            break

    elapsed = time.time() - t0
    PNP_RECORDER.close_episode(success, step + 1, elapsed)
    return success, step + 1, elapsed

Uncertainty Computation

In [ ]:
# ── Phase: Uncertainty measurement ─────────────────────────────────────────
import hashlib, time
from tqdm.notebook import tqdm

SUITES     = ['libero_object_temp_x0.1', 'libero_object_temp_y0.1', 'libero_object_temp_x0.2', 'libero_object_temp_y0.2', 'libero_spatial_with_milk', 'libero_goal_with_yellow_book']
# SUITES     = ['libero_spatial_with_milk', 'libero_goal_with_yellow_book']

N_EPISODES = 10
VERBOSE    = True

PNP_CONFIG.enabled        = True
PNP_CONFIG.mode           = 'uncertainty'
PNP_CONFIG.step_indices   = (3, 4)
PNP_CONFIG.num_iterations = 7

print(f'Suites:    {SUITES}')
print(f'Episodes:  {N_EPISODES}/task')
print(f'P&P mode:  {PNP_CONFIG.mode}')

# Verify
assert hasattr(policy.model, '_orig_sample_actions'), \
    "ERROR: monkey-patch not applied — rerun Section 11"
assert policy.model.sample_actions != policy.model._orig_sample_actions, \
    "ERROR: sample_actions not patched"
assert PNP_CONFIG.enabled, "ERROR: PNP_CONFIG.enabled is False"

import types
is_patched = isinstance(policy.model.sample_actions, types.MethodType) and \
             policy.model.sample_actions.__func__.__name__ == '_sample_actions_pnp'
assert is_patched, "ERROR: sample_actions is not _sample_actions_pnp"

print(f'P&P active: mode={PNP_CONFIG.mode}, steps={PNP_CONFIG.step_indices}, K={PNP_CONFIG.num_iterations}')

Suites:    ['libero_object_temp_x0.1', 'libero_object_temp_y0.1', 'libero_object_temp_x0.2', 'libero_object_temp_y0.2', 'libero_spatial_with_milk', 'libero_goal_with_yellow_book']
Episodes:  10/task
P&P mode:  uncertainty
P&P active: mode=uncertainty, steps=(3, 4), K=7


In [ ]:
def _hash_init_state(init_state) -> str:
    arr = np.asarray(init_state)
    return hashlib.md5(arr.tobytes()).hexdigest()[:16]

all_results    = []
PNP_RECORDER.reset()
benchmark_dict = _bm.get_benchmark_dict()
total_start    = time.time()

for SUITE in tqdm(SUITES, desc='Suites'):
    print(f'\n{"="*60}\nSuite: {SUITE}\n{"="*60}')

    task_suite    = benchmark_dict[SUITE]()
    max_steps     = MAX_STEPS_MAP.get(SUITE, 300)
    suite_results = []
    suite_start   = time.time()

    for task_idx in tqdm(range(task_suite.n_tasks),
                         desc=f'  {SUITE} tasks', leave=False):
        task        = task_suite.get_task(task_idx)
        init_states = task_suite.get_task_init_states(task_idx)

        bddl_file_path = os.path.join(
            get_libero_path('bddl_files'), task.problem_folder, task.bddl_file)

        env = OffScreenRenderEnv(
            bddl_file_name=bddl_file_path, camera_names=CAMERAS,
            camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
            has_offscreen_renderer=True, use_camera_obs=True,
            has_renderer=False, reward_shaping=False,
        )

        n_success = 0
        ep_bar    = tqdm(range(min(N_EPISODES, len(init_states))),
                         desc=f'    T{task_idx+1} ({task.language[:40]}...)',
                         leave=False)

        for ep_idx in ep_bar:
            init_state      = init_states[ep_idx]
            init_state_hash = _hash_init_state(init_state)

            success, n_steps, elapsed = run_episode_pnp(
                env, init_state, policy, task.language, max_steps, device,
                meta={
                    'suite': SUITE, 'task_idx': task_idx,
                    'episode_idx': ep_idx,
                    'init_state_hash': init_state_hash,
                })

            n_success += int(success)

            ep_record = PNP_RECORDER.episodes[-1]
            write_episode_to_db(
                ep_record, SUITE, task_idx, ep_idx,
                init_state_hash, elapsed)

            result = {
                'rollout_id':       ep_record['rollout_id'],
                'suite':            SUITE,
                'task_idx':         task_idx,
                'task_description': task.language,
                'episode':          ep_idx,
                'init_state_hash':  init_state_hash,
                'success':          success,
                'n_steps':          n_steps,
                'elapsed_s':        round(elapsed, 2),
                'pnp_enabled':      PNP_CONFIG.enabled,
                'u_mean_episode':   ep_record['u_mean_episode'],
            }
            suite_results.append(result)
            all_results.append(result)

            status = '✓' if success else '✗'
            u_str  = (f' | U={ep_record["u_mean_episode"]:.4f}'
                      if ep_record['u_mean_episode'] is not None else '')
            ep_bar.set_postfix(sr=f'{n_success}/{ep_idx+1}', status=status)
            if VERBOSE:
                tqdm.write(f'    {status} ep{ep_idx} | {n_steps} steps | '
                           f'{elapsed:.1f}s{u_str}')

        tqdm.write(f'  T{task_idx+1} SR: {n_success/N_EPISODES:.0%} '
                   f'({n_success}/{N_EPISODES}) — {task.language[:50]}...')
        env.close()

    suite_sr  = sum(r['success'] for r in suite_results) / len(suite_results)
    suite_min = (time.time() - suite_start) / 60
    tqdm.write(f'\n{SUITE} SR: {suite_sr:.1%} | Time: {suite_min:.1f} min')

overall_sr = sum(r['success'] for r in all_results) / len(all_results)
total_min  = (time.time() - total_start) / 60
print(f'\n{"="*60}')
print(f'OVERALL SR: {overall_sr:.1%} '
      f'({sum(r["success"] for r in all_results)}/{len(all_results)})')
print(f'Total time: {total_min:.1f} min')
print(f'DB: {DB_PATH}  '
      f'({con.execute("SELECT COUNT(*) FROM rollouts").fetchone()[0]} total rollouts)')
print(f'{"="*60}')

Suites:   0%|          | 0/6 [00:00<?, ?it/s]


Suite: libero_object_temp_x0.1


  libero_object_temp_x0.1 tasks:   0%|          | 0/10 [00:00<?, ?it/s]

    T1 (pick up the alphabet soup and place it i...):   0%|          | 0/10 [00:00<?, ?it/s]

    ✗ ep0 | 280 steps | 15.7s | U=0.0312
    ✓ ep1 | 179 steps | 10.2s | U=0.0209
    ✗ ep2 | 280 steps | 15.0s | U=0.0169
    ✗ ep3 | 280 steps | 15.6s | U=0.0266
    ✓ ep4 | 189 steps | 10.6s | U=0.0214
    ✗ ep5 | 280 steps | 15.3s | U=0.0436
    ✓ ep6 | 179 steps | 10.5s | U=0.0215
    ✗ ep7 | 280 steps | 15.2s | U=0.0206
    ✗ ep8 | 280 steps | 15.5s | U=0.0362
    ✓ ep9 | 184 steps | 10.5s | U=0.0182
  T1 SR: 40% (4/10) — pick up the alphabet soup and place it in the bask...


...

    ✓ ep0 | 74 steps | 4.8s | U=0.0206
    ✓ ep1 | 72 steps | 4.8s | U=0.0166
    ✓ ep2 | 81 steps | 5.0s | U=0.0148
    ✓ ep3 | 76 steps | 4.9s | U=0.0204
    ✓ ep4 | 71 steps | 4.7s | U=0.0250
    ✓ ep5 | 76 steps | 4.9s | U=0.0228
    ✓ ep6 | 79 steps | 5.0s | U=0.0243
    ✓ ep7 | 79 steps | 5.0s | U=0.0170
    ✗ ep8 | 300 steps | 18.1s | U=0.0483
    ✓ ep9 | 77 steps | 4.9s | U=0.0185
  T8 SR: 90% (9/10) — turn on the stove...


    T9 (put the bowl on the plate...):   0%|          | 0/10 [00:00<?, ?it/s]

    ✓ ep0 | 72 steps | 5.0s | U=0.0202
    ✓ ep1 | 71 steps | 4.9s | U=0.0179
    ✓ ep2 | 75 steps | 4.9s | U=0.0193
    ✓ ep3 | 71 steps | 4.7s | U=0.0191
    ✓ ep4 | 78 steps | 5.1s | U=0.0200
    ✓ ep5 | 74 steps | 5.0s | U=0.0178
    ✓ ep6 | 72 steps | 4.8s | U=0.0188
    ✗ ep7 | 300 steps | 17.4s | U=0.0373
    ✓ ep8 | 73 steps | 5.0s | U=0.0205
    ✓ ep9 | 72 steps | 4.7s | U=0.0179
  T9 SR: 90% (9/10) — put the bowl on the plate...


    T10 (put the wine bottle on the rack...):   0%|          | 0/10 [00:00<?, ?it/s]

    ✓ ep0 | 131 steps | 8.2s | U=0.0368
    ✓ ep1 | 146 steps | 8.7s | U=0.0290
    ✓ ep2 | 122 steps | 7.7s | U=0.0279
    ✓ ep3 | 130 steps | 8.0s | U=0.0330
    ✓ ep4 | 155 steps | 10.0s | U=0.0341
    ✓ ep5 | 134 steps | 8.3s | U=0.0312
    ✓ ep6 | 141 steps | 8.4s | U=0.0330
    ✓ ep7 | 163 steps | 10.4s | U=0.0287
    ✓ ep8 | 149 steps | 8.9s | U=0.0269
    ✓ ep9 | 119 steps | 7.5s | U=0.0300
  T10 SR: 100% (10/10) — put the wine bottle on the rack...

libero_goal_with_yellow_book SR: 88.0% | Time: 16.4 min

OVERALL SR: 48.0% (288/600)
Total time: 142.4 min
DB: /content/drive/MyDrive/cs159_jeff/libero_pro_results/rollouts_jeff_2_k7.db  (600 total rollouts)


Refinement

In [ ]:
# ── Phase: Refinement ───────────────────────────────────────────────────────
import hashlib, time
from tqdm.notebook import tqdm

SUITES     = ['libero_object_temp_x0.1', 'libero_object_temp_y0.1', 'libero_object_temp_x0.2', 'libero_object_temp_y0.2', 'libero_spatial_with_milk', 'libero_goal_with_yellow_book']
N_EPISODES = 10
VERBOSE    = True

PNP_CONFIG.enabled        = True
PNP_CONFIG.mode           = 'both'
PNP_CONFIG.step_indices   = (3, 4)
PNP_CONFIG.num_iterations = 7

print(f'Suites:    {SUITES}')
print(f'Episodes:  {N_EPISODES}/task')
print(f'P&P mode:  {PNP_CONFIG.mode}')

all_results    = []
PNP_RECORDER.reset()
benchmark_dict = _bm.get_benchmark_dict()
total_start    = time.time()

for SUITE in tqdm(SUITES, desc='Suites'):
    print(f'\n{"="*60}\nSuite: {SUITE}\n{"="*60}')

    task_suite    = benchmark_dict[SUITE]()
    max_steps     = MAX_STEPS_MAP.get(SUITE, 300)
    suite_results = []
    suite_start   = time.time()

    for task_idx in tqdm(range(task_suite.n_tasks),
                         desc=f'  {SUITE} tasks', leave=False):
        task        = task_suite.get_task(task_idx)
        init_states = task_suite.get_task_init_states(task_idx)

        bddl_file_path = os.path.join(
            get_libero_path('bddl_files'), task.problem_folder, task.bddl_file)

        env = OffScreenRenderEnv(
            bddl_file_name=bddl_file_path, camera_names=CAMERAS,
            camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
            has_offscreen_renderer=True, use_camera_obs=True,
            has_renderer=False, reward_shaping=False,
        )

        n_success = 0
        ep_bar    = tqdm(range(min(N_EPISODES, len(init_states))),
                         desc=f'    T{task_idx+1} ({task.language[:40]}...)',
                         leave=False)

        for ep_idx in ep_bar:
            init_state      = init_states[ep_idx]
            init_state_hash = _hash_init_state(init_state)

            success, n_steps, elapsed = run_episode_pnp(
                env, init_state, policy, task.language, max_steps, device,
                meta={
                    'suite': SUITE, 'task_idx': task_idx,
                    'episode_idx': ep_idx,
                    'init_state_hash': init_state_hash,
                })

            n_success += int(success)

            ep_record = PNP_RECORDER.episodes[-1]
            write_episode_to_db(
                ep_record, SUITE, task_idx, ep_idx,
                init_state_hash, elapsed)

            result = {
                'rollout_id':       ep_record['rollout_id'],
                'suite':            SUITE,
                'task_idx':         task_idx,
                'task_description': task.language,
                'episode':          ep_idx,
                'init_state_hash':  init_state_hash,
                'success':          success,
                'n_steps':          n_steps,
                'elapsed_s':        round(elapsed, 2),
                'pnp_enabled':      PNP_CONFIG.enabled,
                'u_mean_episode':   ep_record['u_mean_episode'],
            }
            suite_results.append(result)
            all_results.append(result)

            status = '✓' if success else '✗'
            u_str  = (f' | U={ep_record["u_mean_episode"]:.4f}'
                      if ep_record['u_mean_episode'] is not None else '')
            ep_bar.set_postfix(sr=f'{n_success}/{ep_idx+1}', status=status)
            if VERBOSE:
                tqdm.write(f'    {status} ep{ep_idx} | {n_steps} steps | '
                           f'{elapsed:.1f}s{u_str}')

        tqdm.write(f'  T{task_idx+1} SR: {n_success/N_EPISODES:.0%} '
                   f'({n_success}/{N_EPISODES}) — {task.language[:50]}...')
        env.close()

    suite_sr  = sum(r['success'] for r in suite_results) / len(suite_results)
    suite_min = (time.time() - suite_start) / 60
    tqdm.write(f'\n{SUITE} SR: {suite_sr:.1%} | Time: {suite_min:.1f} min')

overall_sr = sum(r['success'] for r in all_results) / len(all_results)
total_min  = (time.time() - total_start) / 60
print(f'\n{"="*60}')
print(f'OVERALL SR: {overall_sr:.1%} '
      f'({sum(r["success"] for r in all_results)}/{len(all_results)})')
print(f'Total time: {total_min:.1f} min')
print(f'DB: {DB_PATH}  '
      f'({con.execute("SELECT COUNT(*) FROM rollouts").fetchone()[0]} total rollouts)')
print(f'{"="*60}')

Suites:    ['libero_object_temp_x0.1', 'libero_object_temp_y0.1', 'libero_object_temp_x0.2', 'libero_object_temp_y0.2', 'libero_spatial_with_milk', 'libero_goal_with_yellow_book']
Episodes:  10/task
P&P mode:  both


Suites:   0%|          | 0/6 [00:00<?, ?it/s]


Suite: libero_object_temp_x0.1


  libero_object_temp_x0.1 tasks:   0%|          | 0/10 [00:00<?, ?it/s]

    T1 (pick up the alphabet soup and place it i...):   0%|          | 0/10 [00:00<?, ?it/s]